# Chapter 13 — SIEM과 이벤트 상관 분석 (실습)

## 실습 목표
1. 다중 소스(방화벽·웹·SSH) 정규화
2. 시계열 윈도우 (brute force)
3. 시퀀스 매칭 (실패→성공)
4. 알람 그룹핑

In [ ]:
# 파트별 노트북은 같은 종류의 공통 데이터·모델 작업 폴더를 사용합니다.
from pathlib import Path as _BookPath
import os as _book_os
_book_cwd = _BookPath.cwd()
if (_book_cwd.name in {"part1", "part2", "part3", "part4"}
        and _book_cwd.parent.name in {"instructor", "exercises", "solutions"}):
    _book_os.chdir(_book_cwd.parent)


In [ ]:
# 첫 사용시에만 주석 해제하여 실행 (Colab/venv 환경에 따라 일부는 이미 설치됨)
# !pip install -q pandas
# !pip install -q numpy

In [ ]:
import pandas as pd
import numpy as np
rng = np.random.default_rng(42)

In [ ]:
# === 한글 폰트 설정 (matplotlib) ===
from pathlib import Path
import matplotlib
from matplotlib import font_manager
for fp in ['/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
           '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf']:
    if Path(fp).exists():
        font_manager.fontManager.addfont(fp)
        matplotlib.rcParams['font.family'] = font_manager.FontProperties(fname=fp).get_name()
        break
matplotlib.rcParams['axes.unicode_minus'] = False

---
## Step 1. 다중 소스 → 공통 스키마 정규화

In [ ]:
# ── 실습 과제 ──
# 목표: Step 1. 다중 소스 → 공통 스키마 정규화
# 만들 결과: fw, ssh, SOURCES, src, normalize(), spec
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
# 방화벽 로그 — 시각을 UTC 로 기록하는 장비
fw = ...
    ...

# SSH 로그 — 같은 순간을 한국 시간(+0900) 문자열로 적는 서버
# 10:01:00 UTC == 19:01:00 +0900. 표기만 다르고 실제 시각은 같습니다.
ssh = ...
    ...

# 12장 12.4절에서 만든 normalize 와 같은 규약입니다. 소스를 늘릴 때는 분기만 추가합니다.
SOURCES = ...
    'firewall': dict(ts = ...
    'ssh':      dict(ts = ...
                     src = ...
...

def normalize(df, source: ...
    ...
    파싱 실패는 정상 이벤트로 넘기지 않고 errors = ...
    if source not in SOURCES:
        raise ...
    spec = ...
    return ...
        'ts': pd.to_datetime(df[spec['ts']], format = ...
        ...

events = ...
                   ignore_index = ...

# 정규화를 건너뛰면 두 소스가 9시간 어긋나 같은 시간창에 묶이지 않습니다.
...
```
</details>


---
## Step 2. 시계열 윈도우 — Brute Force

In [ ]:
# ── 실습 과제 ──
# 목표: Step 2. 시계열 윈도우 — Brute Force
# 만들 결과: ssh_evt, rolling, brute
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
ssh_evt = ...
rolling = ...
brute = ...
...
```
</details>


---
## Step 3. 시퀀스 매칭 — 실패 후 성공

In [ ]:
# ── 실습 과제 ──
# 목표: Step 3. 시퀀스 매칭 — 실패 후 성공
# 만들 결과: detect_brute_then_success(), df, alerts, g, fails, succs
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
def detect_brute_then_success(df, window='2min', threshold=5):
    df = ...
    alerts = ...
    for src, g in df.groupby('src'):
        g = ...
        fails = ...
        succs = ...
        for t in succs.index:
            n = ...
            if n >= threshold:
                ...
    return ...

alerts = ...
...
```
</details>


---
## Step 4. 알람 그룹핑

In [ ]:
# ── 실습 과제 ──
# 목표: Step 4. 알람 그룹핑
# 만들 결과: summary, correlate_multi_source(), a, b, events, alerts
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.


<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
# 같은 src 가 firewall blocked + ssh failure 둘 다 발생 → CRITICAL
summary = ...
    ...
def correlate_multi_source(fw, ssh, window='1min'):
    a = ...
    b = ...
    events = ...
    alerts = ...
    for src, group in events.groupby('src'):
        # 같은 시각의 이벤트를 합쳐 두 소스에 동일한 (t-window, t]를 적용한다.
        counts = ...
        counts = ...
        hits = ...
        if not hits.empty:
            t = ...
            ...
    return ...
fw_events = ...
ssh_events = ...
correlated = ...
summary['same_window'] = ...
...
# summary는 전체 기간 빈도이며, 시간 상관 판정은 correlated만 사용한다.
```
</details>


In [ ]:
# 출발지별 이벤트 상관 — 방화벽 차단 + SSH 실패가 함께 튀는 src 가 위험
import matplotlib.pyplot as plt
import numpy as np
srcs = list(summary.index); x = np.arange(len(srcs)); w = 0.35
fig, ax = plt.subplots(figsize=(6, 3.3))
ax.bar(x - w/2, summary['fw_blocked'], w, label='방화벽 차단', color='#1a5f7a')
ax.bar(x + w/2, summary['ssh_failure'], w, label='SSH 실패', color='#e74c3c')
ax.set_xticks(x); ax.set_xticklabels(srcs, rotation=10)
ax.set_title('출발지별 이벤트 상관 (다중 소스 결합)')
ax.set_ylabel('이벤트 수'); ax.legend()
plt.tight_layout(); plt.show()

---
## 정리

- 다중 소스를 같은 스키마로 정규화하는 것이 SIEM 의 핵심.
- pandas rolling 으로 시계열 임계값, 직접 함수로 시퀀스 매칭.
- 다음 챕터에서 본격 지도학습 모델로 분류한다 (ch14).